# Phase 6 — Run and validate the Ecuador 2022 network

This notebook is the student-facing full-year workflow. It performs artifact preflight, loads the Phase 5 candidate, solves all 8,760 hourly snapshots with HiGHS, calculates the annual energy mix and energy-balance diagnostics, and writes the validation tables, JSON summary, report input, and figures. The solve is intentionally manual because it is a scientific run, not a CI test.

## Reproducibility contract

Run this notebook from the repository root in the approved `pypsa-earth-ec` environment. Do not substitute a different weather year, reference table, solver, or network. If preflight fails, stop before scientific processing. The criteria are documented in `docs/PHASE_06_VALIDATION_CRITERIA.md`; configured thresholds are exploratory engineering screens, not calibration targets.

In [ ]:
from pathlib import Path
import hashlib, json
import matplotlib.pyplot as plt
import pandas as pd
import pypsa

ROOT = Path.cwd()
while not (ROOT / 'src').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / 'config/artifacts.yaml').is_file(), 'Run from the repository or a child directory'
import sys
sys.path.insert(0, str(ROOT))
from src.artifacts import run_preflight, format_preflight_report
from src.validation import (load_validation_contract, load_raw_supply_reference,
    build_supply_reference, calculate_validation_metrics, screen_validation,
    validate_annual_coverage, enforce_historical_import_energy)

preflight = run_preflight('06_validation')
print(format_preflight_report(preflight))
preflight.raise_for_errors()

## Load and solve the complete annual candidate

The candidate is the regenerated Phase 5 product containing the reviewed La Concordia-Pedernales and Minas-San Ildefonso topology repairs, documented Colombia interfaces, and separate Colombia/Peru export sinks. Historical mode constrains annual imports to 465.83 GWh within the configured 5% input band; future scenarios must disable that historical energy constraint. The solver status and termination condition are retained in the run metadata.

In [ ]:
candidate_path = ROOT / 'data/processed/networks/network_base_filled.nc'
result_dir = ROOT / 'results/phase_06'
figure_dir = ROOT / 'reports/figures/phase_06'
result_dir.mkdir(parents=True, exist_ok=True); figure_dir.mkdir(parents=True, exist_ok=True)
network = pypsa.Network(candidate_path)
contract = load_validation_contract(ROOT / 'config/validation_2022.yaml')
# Enforce the documented 450 MW Colombia interface ceiling before solving.
colombia_generators = network.generators.index[network.generators.carrier.eq('interconnection_colombia')]
if len(colombia_generators): network.generators.loc[colombia_generators, 'p_nom'] = 450.0
colombia_links = network.links.index[network.links.carrier.eq('interconnection_colombia')]
if len(colombia_links): network.links.loc[colombia_links, 'p_nom'] *= 450.0 / network.links.loc[colombia_links, 'p_nom'].sum()
network.meta = dict(network.meta or {}); network.meta['colombia_export_cap_mw'] = 450.0
comparable, reasons = validate_annual_coverage(network.snapshots, 2022)
assert comparable, reasons
reasons.append('CENACE production and demand totals retain an unresolved 931.43 GWh metering-boundary residual; total-energy screening is disabled and the run remains PARTIAL.')
solver_status, termination = network.optimize(
    solver_name='highs',
    extra_functionality=lambda n, snapshots: enforce_historical_import_energy(n, snapshots, contract),
)
print({'solver_status': str(solver_status), 'termination_condition': str(termination)})
solved_path = result_dir / 'network_solved_2022.nc'
# Release the large Linopy/HiGHS model before serializing the solved network.
network.model = None
import gc; gc.collect()
network.export_to_netcdf(solved_path)

## Annual supply and validation metrics

Positive generator dispatch is aggregated through the configured carrier mapping. Load shedding is reported separately and never counted as actual supply.

In [ ]:
raw = load_raw_supply_reference(ROOT / contract['supply_reference']['raw_table'])
reference = build_supply_reference(raw, contract)
dispatch = network.generators_t.p.clip(lower=0)
weighted_gwh = dispatch.mul(network.snapshot_weightings.generators, axis=0).sum() / 1000
by_carrier = weighted_gwh.groupby(network.generators.loc[weighted_gwh.index, 'carrier']).sum()
load_shedding_gwh = float(by_carrier.get('load_shedding', 0.0))
by_carrier = by_carrier.drop(labels=['load_shedding'], errors='ignore')
domestic_ids = network.loads.index[network.loads.carrier.eq('electricity')]
export_ids = network.loads.index[network.loads.carrier.eq('export')]
domestic_demand_gwh = float(network.loads_t.p_set.loc[:, domestic_ids].mul(network.snapshot_weightings.objective, axis=0).sum().sum() / 1000)
exports_gwh = float(network.loads_t.p_set.loc[:, export_ids].mul(network.snapshot_weightings.objective, axis=0).sum().sum() / 1000)
energy_balance_residual_gwh = float(by_carrier.sum() + load_shedding_gwh - domestic_demand_gwh - exports_gwh)
from src.validation import aggregate_model_supply
simulated = aggregate_model_supply(by_carrier, contract)
rows, metrics = calculate_validation_metrics(simulated, reference)
screen = screen_validation(metrics, rows, contract['acceptance'])
validation_csv = result_dir / 'validation_2022.csv'; rows.to_csv(validation_csv, index=False)
summary = {'status': 'FAIL' if not screen['passed'] else 'PARTIAL', 'metrics': metrics, 'screening': screen,
           'comparability': {'status': 'PARTIAL', 'reasons': reasons},
           'load_shedding_gwh': load_shedding_gwh,
           'domestic_demand_gwh': domestic_demand_gwh, 'exports_gwh': exports_gwh,
           'energy_balance_residual_gwh': energy_balance_residual_gwh,
           'unresolved_accounting_residual': contract['accounting_residual'],
           'solver_status': str(solver_status),
           'termination_condition': str(termination), 'reference_source': contract['sources']['CENACE_ANNUAL_2022']}
(result_dir / 'validation_summary.json').write_text(json.dumps(summary, indent=2, default=str), encoding='utf-8')
display(rows); print(json.dumps(summary, indent=2, default=str))

## Figures and handoff

The three figures are saved for the Phase 6 report. Review energy balance residuals, exports, storage flows, curtailment, and load shedding before treating the result as scientifically accepted.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4)); rows.set_index('carrier')[['reference_gwh', 'simulated_gwh']].plot.bar(ax=ax); ax.set_ylabel('GWh'); fig.tight_layout(); fig.savefig(figure_dir / 'reference_vs_model_gwh.png', dpi=160); plt.show()
fig, ax = plt.subplots(figsize=(8, 4)); rows.set_index('carrier')['signed_error_pp'].plot.bar(ax=ax); ax.axhline(0, color='black', lw=.8); ax.set_ylabel('percentage points'); fig.tight_layout(); fig.savefig(figure_dir / 'carrier_percentage_point_errors.png', dpi=160); plt.show()
fig, ax = plt.subplots(figsize=(8, 4)); rows.set_index('carrier')[['reference_share', 'simulated_share']].plot.bar(ax=ax); ax.set_ylabel('share'); fig.tight_layout(); fig.savefig(figure_dir / 'reference_vs_model_mix.png', dpi=160); plt.show()
artifact = {'network': str(solved_path.relative_to(ROOT)), 'sha256': hashlib.sha256(solved_path.read_bytes()).hexdigest(), 'validation_csv': str(validation_csv.relative_to(ROOT)), 'status': 'manual Phase 6 run output'}
(result_dir / 'artifacts.json').write_text(json.dumps(artifact, indent=2), encoding='utf-8')
print(artifact)
print('Canonical Phase 06H solved baseline written; no candidate selector is used.')